# LoRA Compress - Adaptive Autoresearch with OATFeatures intelligent exploration:- **Two-region LR**: 70% conservative (0.0005-0.1), 30% aggressive (0.1-0.8)- **OAT**: Doubles LR until error rises, then refines downward 5% at a time- **Adaptive**: Learns from results to focus on promising regions

In [ ]:
from google.colab import driveimport osdrive.mount('/content/drive')DRIVE_BASE = '/content/drive/MyDrive/LoRA_Compress'os.makedirs(DRIVE_BASE, exist_ok=True)os.makedirs(f'{DRIVE_BASE}/databases', exist_ok=True)os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)

In [ ]:
%cd /content!git clone https://github.com/qades/loracompress.git%cd loracompress!pip install -q transformers torch optuna tqdm

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport optunaimport randomfrom transformers import AutoModelForCausalLMclass AdaptiveExplorer:    def __init__(self, lr_min=0.0005, lr_max=0.8, rank_min=32, rank_max=256, aggressive_ratio=0.3):        self.lr_min = lr_min        self.lr_max = lr_max        self.rank_min = rank_min        self.rank_max = rank_max        self.aggressive_ratio = aggressive_ratio        self.lr_history = []        self.lr_upper_boundary = None        self.lr_optimal_region = None        self.oat_phase = 'explore_lr_up'        self.oat_test_values = [0.01, 0.02, 0.04, 0.08, 0.16, 0.32, 0.64]        self.oat_test_idx = 0        self.oat_best_lr = None        def sample_lr_two_region(self, trial):        if random.random() < self.aggressive_ratio:            return trial.suggest_float('lr', 0.1, self.lr_max, log=True)        return trial.suggest_float('lr', self.lr_min, 0.1, log=True)        def sample_lr_adaptive(self, trial):        if self.lr_optimal_region and random.random() < 0.6:            low, high = self.lr_optimal_region            return trial.suggest_float('lr', max(self.lr_min, low*0.8), min(self.lr_max, high*1.2), log=True)        return self.sample_lr_two_region(trial)        def update_from_result(self, lr, error):        self.lr_history.append((lr, error))        if len(self.lr_history) >= 5:            sorted_lrs = sorted(self.lr_history, key=lambda x: x[0])            best_err = min(e for _, e in sorted_lrs)            good_lrs = [lr for lr, err in sorted_lrs if err < best_err * 1.5]            if good_lrs:                self.lr_optimal_region = (min(good_lrs), max(good_lrs))        def get_oat_suggestion(self, trial_number):        if trial_number == 0:            return {'rank': 64, 'lr': 0.01, 'epochs': 500}                if self.oat_phase == 'explore_lr_up':            if len(self.lr_history) >= 2 and self.lr_history[-1][1] > self.lr_history[-2][1] * 1.3:                self.lr_upper_boundary = self.lr_history[-1][0]                self.oat_best_lr = self.lr_history[-2][0]                self.oat_phase = 'refine_lr_down'                self.oat_test_values = [self.oat_best_lr * (0.95 ** i) for i in range(1, 10)]                self.oat_test_idx = 0                print(f'  [OAT] Upper boundary at {self.lr_upper_boundary:.4f}, refining down')                        if self.oat_test_idx < len(self.oat_test_values):                lr = self.oat_test_values[self.oat_test_idx]                self.oat_test_idx += 1                return {'rank': 64, 'lr': lr, 'epochs': 500}                if self.oat_phase == 'refine_lr_down':            if len(self.lr_history) >= 2 and self.lr_history[-1][1] > self.lr_history[-2][1]:                self.oat_best_lr = self.lr_history[-2][0]                self.oat_phase = 'explore_rank'                self.oat_test_values = [32, 48, 64, 96, 128, 192, 256]                self.oat_test_idx = 0                print(f'  [OAT] Best LR: {self.oat_best_lr:.4f}, exploring ranks')                        if self.oat_test_idx < len(self.oat_test_values):                lr = self.oat_test_values[self.oat_test_idx]                self.oat_test_idx += 1                return {'rank': 64, 'lr': lr, 'epochs': 500}                if self.oat_phase == 'explore_rank':            if self.oat_test_idx < len(self.oat_test_values):                rank = self.oat_test_values[self.oat_test_idx]                self.oat_test_idx += 1                return {'rank': rank, 'lr': self.oat_best_lr or 0.01, 'epochs': 500}            self.oat_phase = 'exploit'                return Nonedef compute_l1_error(W_approx, target):    l1_error = torch.mean(torch.abs(W_approx - target)).item()    mean_abs_target = torch.mean(torch.abs(target)).item()    return (l1_error / mean_abs_target * 100) if mean_abs_target > 0 else float('inf')def train_lora(target_weight, rank, lr, epochs, device='cuda'):    d, k = target_weight.shape    target = target_weight.float().to(device)    A = nn.Parameter(torch.randn(rank, k, device=device) * 0.01)    B = nn.Parameter(torch.randn(d, rank, device=device) * 0.01)    optimizer = torch.optim.AdamW([A, B], lr=lr)        best_loss = float('inf')    best_A, best_B = None, None    epochs_no_improve = 0        for epoch in range(epochs):        optimizer.zero_grad()        W_approx = torch.matmul(B, A)        loss = F.mse_loss(W_approx, target)                if not torch.isfinite(loss):            return float('inf'), 0                loss.backward()        optimizer.step()                current = loss.item()        if current < best_loss - 1e-9:            best_loss = current            best_A = A.detach().clone()            best_B = B.detach().clone()            epochs_no_improve = 0        else:            epochs_no_improve += 1                if epoch >= 200 and epochs_no_improve >= 100:            break        with torch.no_grad():        W_best = torch.matmul(best_B, best_A)        l1_error = compute_l1_error(W_best, target)        return l1_error, epoch + 1print('Functions loaded')

In [ ]:
MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M'LAYER_PATTERN = 'layers.15'print('Loading model...')model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)target_weight = Nonelayer_name = Nonefor name, param in model.named_parameters():    if LAYER_PATTERN in name and len(param.shape) == 2:        target_weight = param.data        layer_name = name        breakprint(f'Layer: {layer_name}')print(f'Shape: {target_weight.shape}')device = 'cuda' if torch.cuda.is_available() else 'cpu'target_weight_gpu = target_weight.float().to(device)

In [ ]:
TARGET_QUALITY = 5.0N_TRIALS = 50EXPLORATION_MODE = 'oat'explorer = AdaptiveExplorer(lr_min=0.0005, lr_max=0.8, rank_min=32, rank_max=256)def objective(trial):    trial_num = len(explorer.lr_history)        if EXPLORATION_MODE == 'oat':        oat_config = explorer.get_oat_suggestion(trial_num)        if oat_config:            rank = oat_config['rank']            lr = oat_config['lr']            epochs = oat_config['epochs']            print(f'  [OAT {explorer.oat_phase}] rank={rank}, lr={lr:.4f}')        else:            rank = trial.suggest_int('rank', 32, 256, log=True)            lr = explorer.sample_lr_adaptive(trial)            epochs = trial.suggest_int('epochs', 200, 2000, log=True)    elif EXPLORATION_MODE == 'two_region':        rank = trial.suggest_int('rank', 32, 256, log=True)        lr = explorer.sample_lr_two_region(trial)        epochs = trial.suggest_int('epochs', 200, 2000, log=True)    else:        rank = trial.suggest_int('rank', 32, 256, log=True)        lr = trial.suggest_float('lr', 0.0005, 0.8, log=True)        epochs = trial.suggest_int('epochs', 200, 2000, log=True)        error, actual_epochs = train_lora(target_weight_gpu, rank, lr, epochs, device)    explorer.update_from_result(lr, error)        d, k = target_weight.shape    compression = (d * k) / (rank * (d + k))        if error <= TARGET_QUALITY:        score = -compression * 10    else:        score = 1000 + (error - TARGET_QUALITY) * 100        trial.set_user_attr('error', error)    trial.set_user_attr('compression', compression)    return scoreprint('Ready to run')

In [ ]:
import osstudy_name = f'oat_{TARGET_QUALITY}pct_{LAYER_PATTERN.replace(".", "_")}'db_file = f'{DRIVE_BASE}/databases/{study_name}.db'os.makedirs(os.path.dirname(db_file), exist_ok=True)study = optuna.create_study(    study_name=study_name,    storage=f'sqlite:///{db_file}',    direction='minimize',    load_if_exists=True)print(f'Running {N_TRIALS} trials with {EXPLORATION_MODE} mode')study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)print(f'Best error: {study.best_trial.user_attrs["error"]:.2f}%')print(f'Best config: rank={study.best_params.get("rank")}, lr={study.best_params.get("lr"):.4f}')